# Reproduce: gap-protected spectral invariants on T^d/Z_2

This notebook recomputes the paper's headline numbers from first
principles and checks each one against the value **as printed in the
paper**, with the equation or theorem number next to it. Then it runs the
full verification suite.

Run all cells top to bottom (in Colab: *Runtime -> Run all*).

Paper: *Gap-protected spectral invariants on T^d/Z_2: dimensional rigidity
at d = 4* (doi:10.5281/zenodo.20597196).

Each check below prints four lines:

- **paper** -- where the claim appears (equation / theorem / section),
- **expect** -- the value exactly as the paper states it,
- **found** -- the value recomputed here,
- **match** -- `OK` when they agree to the paper's stated precision.

| # | Claim | Paper location |
|---|-------|----------------|
| 1 | Shell counts r_4(k), k=0..5 | Jacobi four-square; Sec 5 |
| 2 | N_4(5) = 1 + 8 + 128 = 137 | Eq. (2), Thm 5.1 |
| 3 | Three sectors = b_0, chi_orb, Fix*chi_orb | Eq. (2) |
| 4 | Gram eigenvalues lambda_w | Sec 6 table |
| 5 | Scattering eigenvalues mu_w | Sec 6, contraction bound |
| 6 | F_0 (Rankin-Selberg constant) | Sec 7 |
| 7 | Delta_1 = F_0 + gamma_E/2 | Eq. (5) |
| 8 | Smooth action N_4(K*) + Delta_1 | Eq. (6) |
| 9 | Spectral radius rho | Eq. (3) |
| 10 | Born remainder rho^2/(1-rho) | Eq. (4) |
| 11 | Born interval | Sec 7 |


In [ ]:
# Setup: no-op locally; in Colab/fresh Jupyter it fetches the repo and deps.
import os, subprocess, sys
if not os.path.exists('run_all.py'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kootru-repo/'
                    'gap-protected-spectral-invariants', '_repo'], check=True)
    os.chdir('_repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
print('ready in', os.getcwd())


In [ ]:
import itertools, math
import mpmath as mp
mp.mp.dps = 30

# ---- reporting helper: expected (paper) vs found (computed) ----
n_ok = n_tot = 0
def report(claim, paper, expect, found, ok):
    global n_ok, n_tot
    n_tot += 1; n_ok += int(bool(ok))
    print(claim)
    print('  paper :', paper)
    print('  expect:', expect)
    print('  found :', found)
    print('  match :', 'OK' if ok else '*** MISMATCH ***')
    print()

# lattice ball |n|^2 <= 5 on Z^4, and the Krawtchouk polynomial K_w(h)
pts = [n for n in itertools.product(range(-3, 4), repeat=4)
       if sum(x*x for x in n) <= 5]
def kraw(h, w, d=4):
    return sum((-1)**j * math.comb(w, j) * math.comb(d - w, h - j)
               for j in range(h + 1) if 0 <= h - j <= d - w)

# ===== 1. shell counts r_4(k) =====
def N4(K):
    return sum(1 for n in pts if sum(x*x for x in n) <= K) if K >= 5 else \
        sum(1 for n in itertools.product(range(-3, 4), repeat=4)
            if sum(x*x for x in n) <= K)
r4 = [N4(k) - (N4(k-1) if k else 0) for k in range(6)]
report('1. Shell counts r_4(k), k = 0..5',
       'Jacobi four-square theorem; Section 5',
       '[1, 8, 24, 32, 24, 48], sum = 137',
       '%s, sum = %d' % (r4, sum(r4)),
       r4 == [1, 8, 24, 32, 24, 48] and sum(r4) == 137)

# ===== 2. mode count and decomposition =====
b0, chi, dyn = r4[0], r4[1], sum(r4[2:6])
report('2. N_4(5) = b_0 + chi_orb + |Fix|*chi_orb',
       'Eq. (2), Theorem 5.1 (Dimensional rigidity)',
       '137 = 1 + 8 + 128',
       '%d = %d + %d + %d' % (b0 + chi + dyn, b0, chi, dyn),
       (b0, chi, dyn) == (1, 8, 128))

# ===== 3. each summand is its own invariant =====
report('3. Sector identities r_4(1)=chi_orb and dyn=|Fix|*chi_orb',
       'Eq. (2) underbraces',
       'chi_orb = 8, |Fix|*chi_orb = 16*8 = 128',
       'chi_orb = %d, |Fix|*chi_orb = 16*%d = %d' % (chi, chi, 16*chi),
       chi == 8 and 16*chi == dyn == 128)

# ===== 4. integer Gram eigenvalues lambda_w =====
# distance-h character sums g_h = sum_{ball} (-1)^{n_1+...+n_h}
g = [sum((-1)**sum(n[k] for k in range(h)) for n in pts) for h in range(5)]
lam = [int(sum(kraw(h, w) * g[h] for h in range(5))) for w in range(5)]
report('4. Gram eigenvalues lambda_w (w = 0..4)',
       'Section 6, eigenvalue table',
       '[144, 224, 64, 128, 256]   (g_0 = N_4(5) = 137)',
       '%s   (g_0 = %d)' % (lam, g[0]),
       lam == [144, 224, 64, 128, 256] and g[0] == 137)

# ===== 5. continuous scattering eigenvalues mu_w =====
pi, gE, ln2 = mp.pi, mp.euler, mp.log(2)
Z = [-8*ln2, (pi - 2*ln2)/2, -2*ln2, (-pi - 2*ln2)/2, -4*ln2]  # theta sums
G = [z/(4*pi**2) for z in Z]
mu = [sum(G[h]*kraw(h, w) for h in range(5)) for w in range(5)]
mu_paper = [-0.562, 0.089, -0.140, -0.229, -0.281]
report('5. Scattering-kernel eigenvalues mu_w (w = 0..4)',
       'Section 6, contraction bound (SI Prop. born-bound)',
       '(-0.562, 0.089, -0.140, -0.229, -0.281)',
       '(' + ', '.join(mp.nstr(m, 3) for m in mu) + ')',
       all(abs(float(m) - p) < 5e-4 for m, p in zip(mu, mu_paper)))

# ===== 6. Rankin-Selberg constant F_0 =====
F0 = 1 - mp.log(pi) + 6*mp.zeta(2, 1, 1)/pi**2 + mp.log(4)/3
report('6. F_0 = 1 - ln(pi) + 6 zeta\'(2)/pi^2 + ln(4)/3',
       'Section 7 (lattice invariant of the Z^4 theta function)',
       '-0.252592758...',
       mp.nstr(F0, 10),
       abs(F0 - mp.mpf('-0.252592758')) < 1e-9)

# ===== 7. Delta_1 =====
D1 = F0 + gE/2
report('7. Delta_1 = F_0 + gamma_E/2',
       'Eq. (5)',
       '0.0360151',
       mp.nstr(D1, 8),
       abs(D1 - mp.mpf('0.0360151')) < 5e-7)

# ===== 8. smooth spectral action =====
action = 137 + D1
report('8. Smooth spectral action  N_4(K*) + Delta_1',
       'Eq. (6)',
       '137.036015074...',
       mp.nstr(action, 12),
       abs(action - mp.mpf('137.036015074')) < 5e-10)

# ===== 9. spectral radius rho =====
rho = max(abs(m) for m in mu)/(8*pi**2)
report('9. Spectral radius  rho = max_w |mu_w| / (8 pi^2)',
       'Eq. (3)',
       'rho < 7.2e-3',
       mp.nstr(rho, 5),
       rho < mp.mpf('7.2e-3'))

# ===== 10. Born remainder =====
rem = rho**2/(1 - rho)
report('10. Born remainder  rho^2/(1-rho)',
       'Eq. (4)',
       '< 5.3e-5',
       mp.nstr(rem, 5),
       rem < mp.mpf('5.3e-5'))

# ===== 11. Born interval =====
lo, hi = action - rem, action + rem
report('11. Born interval  [N_4(K*)+Delta_1 -+ rho^2/(1-rho)]',
       'Section 7',
       'contained in [137.03596, 137.03607]',
       '[%s, %s]' % (mp.nstr(lo, 9), mp.nstr(hi, 9)),
       lo >= mp.mpf('137.03596') and hi <= mp.mpf('137.03607'))

print('=' * 60)
print('headline claims reproduced: %d / %d' % (n_ok, n_tot))
assert n_ok == n_tot, 'a headline claim did not match the paper'


## Full verification suite

The cell below runs every script in the repository -- the same command the
continuous-integration workflow runs (the 5 primary scripts, the
adversarial battery, the crystallographic sweeps, and the definitive
value recomputation). It exits cleanly only if all checks pass.


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'run_all.py'], capture_output=True, text=True)
print(r.stdout[-2500:])
print('exit code:', r.returncode)
